# Inferential Statistics — Tutorial

**Time:** 60–90 minutes

**Primary outcome:** Apply population/sample reasoning, construct confidence intervals, calculate p-values, and interpret the Central Limit Theorem using real restaurant data.

## What you will learn

| Section | Concept |
|---|---|
| 1 | Population vs Sample |
| 2 | Central Limit Theorem |
| 3 | Real Data: The Tips Dataset |
| 4 | Confidence Intervals |
| 5 | Standard Error and Sample Size |
| 6 | Hypothesis Testing: The Logic |
| 7 | One-Sample t-Test |
| 8 | Two-Sample t-Test |
| 9 | Chi-Square Test of Independence |
| 10 | Practice Exercises |

> **Inferential statistics** lets us draw conclusions about a *population* from a *sample*. We never see the full picture — we reason under uncertainty.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

print("Libraries loaded successfully.")

---
## 1. Population vs Sample

A **population** is the complete set of all items we care about.  
A **sample** is a smaller subset we actually observe.

**Key vocabulary:**

| Term | Population | Sample |
|---|---|---|
| Mean | μ (mu) — *parameter* | x̄ (x-bar) — *statistic* |
| Std dev | σ (sigma) | s |
| Size | N | n |

We will generate a synthetic population of 10,000 restaurant bills, then draw samples of different sizes and see how close each sample mean comes to the true population mean.

In [ ]:
# Generate synthetic population: 10,000 restaurant bills
population = np.random.normal(loc=20.0, scale=8.0, size=10_000)
population = np.clip(population, 3, 80)  # bills can't be negative or absurdly large

pop_mean = population.mean()
pop_std  = population.std()
print(f"Population  N={len(population):,}")
print(f"True mean   μ = ${pop_mean:.2f}")
print(f"True std    σ = ${pop_std:.2f}")

In [ ]:
# Draw three samples of different sizes
sample_30  = np.random.choice(population, size=30,  replace=False)
sample_100 = np.random.choice(population, size=100, replace=False)
sample_300 = np.random.choice(population, size=300, replace=False)

fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=False)

datasets = [
    (population,  'Population (N=10,000)', 60),
    (sample_30,   'Sample n=30',           15),
    (sample_100,  'Sample n=100',          20),
    (sample_300,  'Sample n=300',          30),
]

for ax, (data, title, bins) in zip(axes, datasets):
    ax.hist(data, bins=bins, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(data.mean(), color='crimson', linewidth=2,
               label=f'Mean = ${data.mean():.2f}')
    ax.axvline(pop_mean, color='orange', linewidth=2, linestyle='--',
               label=f'True μ = ${pop_mean:.2f}')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Bill ($)')
    ax.legend(fontsize=8)

plt.suptitle('Sample mean converges to population mean as n grows', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("\nHow far off is each sample mean from the true population mean?")
for n, s in [(30, sample_30), (100, sample_100), (300, sample_300)]:
    print(f"  n={n:>3}: x̄ = ${s.mean():.2f}  |error| = ${abs(s.mean() - pop_mean):.2f}")

**Takeaway:** As sample size grows, the sample mean gets closer to the population mean. But even a small sample of n=30 can give a reasonable estimate — and that is the entire point of inferential statistics.

---
## 2. The Central Limit Theorem (CLT)

> **Central Limit Theorem:** No matter the shape of the *population* distribution, the distribution of *sample means* (the sampling distribution) will be approximately **normal** when n is large enough (typically n ≥ 30).

To really show this, we will deliberately start with a **right-skewed** population (exponential distribution) — definitely not normal — and watch what happens to the distribution of sample means.

In [ ]:
# Skewed population: exponential (e.g., waiting times, insurance claims)
skewed_pop = np.random.exponential(scale=10, size=100_000)

# Take 1,000 samples of n=30 and record each sample mean
n_samples   = 1000
sample_size = 30
sample_means = [
    np.random.choice(skewed_pop, size=sample_size, replace=False).mean()
    for _ in range(n_samples)
]
sample_means = np.array(sample_means)

# Theoretical standard error
theoretical_se = skewed_pop.std() / np.sqrt(sample_size)
observed_se    = sample_means.std()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: the skewed population
axes[0].hist(skewed_pop, bins=80, color='salmon', edgecolor='white', alpha=0.8)
axes[0].set_title('Population Distribution\n(Exponential — clearly right-skewed)', fontsize=11)
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Frequency')

# Right: sampling distribution of means
axes[1].hist(sample_means, bins=40, color='mediumseagreen', edgecolor='white', alpha=0.8, density=True)
# Overlay a normal curve for reference
x = np.linspace(sample_means.min(), sample_means.max(), 300)
axes[1].plot(x, stats.norm.pdf(x, sample_means.mean(), sample_means.std()),
             color='darkgreen', linewidth=2.5, label='Normal curve')
axes[1].set_title(f'Sampling Distribution of Means\n(1,000 samples, n={sample_size} each)', fontsize=11)
axes[1].set_xlabel('Sample Mean')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.suptitle('CLT: Skewed population → Normal sampling distribution', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f"Theoretical SE = σ/√n = {theoretical_se:.4f}")
print(f"Observed std of sample means   = {observed_se:.4f}")
print(f"These should be very close — and they are!")

**Why does this matter?**

Because almost every hypothesis test and confidence interval formula assumes normality of the *sampling distribution* (not the raw data). The CLT is the mathematical guarantee that makes this assumption valid — as long as n is large enough.

---
## 3. Real Data: The Tips Dataset

We will now work with a real dataset: tips recorded at a restaurant over several months. We will **treat the full dataset as our population**, then explore what happens when we only observe a small sample.

In [ ]:
tips = sns.load_dataset('tips')

print("Shape:", tips.shape)
print("\nColumn types:")
print(tips.dtypes)
print("\nFirst 5 rows:")
tips.head()

In [ ]:
print("Descriptive statistics for numeric columns:")
tips[['total_bill', 'tip', 'size']].describe().round(2)

In [ ]:
# Treat the full tips dataset as the population
true_mean_bill = tips['total_bill'].mean()
print(f"True population mean total_bill: μ = ${true_mean_bill:.2f}")

# Draw 5 random samples of n=30 and compare their means to the true mean
fig, ax = plt.subplots(figsize=(10, 5))

sample_means_tips = []
for i in range(5):
    s = tips['total_bill'].sample(n=30, random_state=i)
    sample_means_tips.append(s.mean())

ax.axhline(true_mean_bill, color='crimson', linewidth=2,
           label=f'True μ = ${true_mean_bill:.2f}', zorder=3)
ax.scatter(range(1, 6), sample_means_tips, s=120, zorder=4, color='steelblue',
           label='Sample means (n=30)')
for i, m in enumerate(sample_means_tips):
    ax.plot([i+1, i+1], [true_mean_bill, m], color='gray', linewidth=1, linestyle='--')
    ax.text(i+1, m + 0.3, f'${m:.2f}', ha='center', fontsize=9)

ax.set_xticks(range(1, 6))
ax.set_xticklabels([f'Sample {i}' for i in range(1, 6)])
ax.set_ylabel('Mean Total Bill ($)')
ax.set_title('Five random samples (n=30) vs the true population mean')
ax.legend()
plt.tight_layout()
plt.show()

print("\nSample means:", [f'${m:.2f}' for m in sample_means_tips])

**The core challenge of statistics:**

In real life we only get *one* sample. We do not know μ. We cannot repeat the experiment 5 times and average the answers. So how do we make a confident claim about the population mean?

That is exactly what **confidence intervals** solve.

---
## 4. Confidence Intervals

A **confidence interval (CI)** gives a *range of plausible values* for the population parameter, based on a single sample.

**Formula (using t-distribution):**

$$\bar{x} \pm t^* \cdot \frac{s}{\sqrt{n}}$$

- $t^*$ is the critical value for the chosen confidence level and degrees of freedom
- $s$ is the sample standard deviation
- $n$ is the sample size

**Interpretation:**  
"If we repeated this sampling procedure 100 times, approximately 95 of the resulting confidence intervals would contain the true population mean."

In [ ]:
# Take one sample of n=50 from the tips dataset
sample_bills = tips['total_bill'].sample(n=50, random_state=42)

# Construct 90%, 95%, 99% confidence intervals
confidence_levels = [0.90, 0.95, 0.99]
ci_results = {}

n    = len(sample_bills)
xbar = sample_bills.mean()
se   = stats.sem(sample_bills)

print(f"Sample: n={n}, x̄ = ${xbar:.2f}, SE = ${se:.2f}\n")
for cl in confidence_levels:
    lo, hi = stats.t.interval(cl, df=n-1, loc=xbar, scale=se)
    ci_results[cl] = (lo, hi)
    print(f"{int(cl*100)}% CI: (${lo:.2f}, ${hi:.2f})  width = ${hi - lo:.2f}")
    print(f"  → We are {int(cl*100)}% confident the true average bill is between ${lo:.2f} and ${hi:.2f}")
    print()

In [ ]:
# Visualise all three CIs as horizontal bars
fig, ax = plt.subplots(figsize=(10, 4))

colors = ['#2ecc71', '#3498db', '#9b59b6']
y_positions = [3, 2, 1]

for (cl, (lo, hi)), color, y in zip(ci_results.items(), colors, y_positions):
    ax.barh(y, hi - lo, left=lo, height=0.4, color=color, alpha=0.7,
            label=f'{int(cl*100)}% CI: (${lo:.2f}, ${hi:.2f})')
    ax.text((lo + hi) / 2, y, f'{int(cl*100)}%', ha='center', va='center',
            fontsize=11, fontweight='bold', color='white')

ax.axvline(xbar, color='black', linewidth=2, linestyle='--', label=f'Sample mean = ${xbar:.2f}')
ax.axvline(true_mean_bill, color='crimson', linewidth=2, label=f'True μ = ${true_mean_bill:.2f}')
ax.set_yticks([])
ax.set_xlabel('Total Bill ($)')
ax.set_title('90%, 95%, and 99% Confidence Intervals\n(wider bar = higher confidence)')
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Simulation: take 100 samples, show which 95% CIs capture the true mean
n_sim   = 100
n_each  = 30
captured = []
intervals = []

for i in range(n_sim):
    s = tips['total_bill'].sample(n=n_each, random_state=i)
    lo, hi = stats.t.interval(0.95, df=n_each-1, loc=s.mean(), scale=stats.sem(s))
    intervals.append((lo, hi))
    captured.append(lo <= true_mean_bill <= hi)

fig, ax = plt.subplots(figsize=(12, 7))
for i, ((lo, hi), hit) in enumerate(zip(intervals, captured)):
    color = '#2ecc71' if hit else '#e74c3c'
    ax.plot([lo, hi], [i, i], color=color, linewidth=1.2, alpha=0.8)

ax.axvline(true_mean_bill, color='black', linewidth=2,
           label=f'True μ = ${true_mean_bill:.2f}', zorder=5)

# Legend proxies
from matplotlib.patches import Patch
ax.legend(handles=[
    ax.lines[-1],
    Patch(color='#2ecc71', label=f'Contains μ ({sum(captured)}/100)'),
    Patch(color='#e74c3c', label=f'Misses μ ({100 - sum(captured)}/100)'),
], fontsize=10)

ax.set_xlabel('Total Bill ($)')
ax.set_ylabel('Simulation number')
ax.set_title(f'100 independent 95% CIs — {sum(captured)}% captured the true mean')
plt.tight_layout()
plt.show()

print(f"\n{sum(captured)} out of 100 CIs captured the true mean.")
print("A 95% CI should capture the true mean ~95% of the time. The CLT + t-distribution deliver!")

---
## 5. Standard Error and Sample Size

The **Standard Error (SE)** measures how much a sample mean typically varies from the true population mean:

$$SE = \frac{\sigma}{\sqrt{n}}$$

Larger samples → smaller SE → more precise estimates. But the gains follow a **diminishing returns** curve.

In [ ]:
sigma = tips['total_bill'].std()
sample_sizes = np.arange(5, 501)
se_values    = sigma / np.sqrt(sample_sizes)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(sample_sizes, se_values, color='steelblue', linewidth=2.5)

# Highlight n=30 as the common rule-of-thumb
se_at_30 = sigma / np.sqrt(30)
ax.axvline(30, color='crimson', linestyle='--', linewidth=1.8, label=f'n=30  SE=${se_at_30:.2f}')
ax.axhline(se_at_30, color='crimson', linestyle=':', linewidth=1.2)
ax.scatter([30], [se_at_30], color='crimson', zorder=5, s=80)

# Highlight n=100
se_at_100 = sigma / np.sqrt(100)
ax.axvline(100, color='darkorange', linestyle='--', linewidth=1.8, label=f'n=100  SE=${se_at_100:.2f}')
ax.scatter([100], [se_at_100], color='darkorange', zorder=5, s=80)

ax.set_xlabel('Sample Size (n)')
ax.set_ylabel('Standard Error ($)')
ax.set_title('Standard Error vs Sample Size — diminishing returns')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f"σ (population std) = ${sigma:.2f}")
print(f"SE at n=5:   ${sigma/np.sqrt(5):.2f}")
print(f"SE at n=30:  ${se_at_30:.2f}  ← CLT kicks in here")
print(f"SE at n=100: ${se_at_100:.2f}")
print(f"SE at n=500: ${sigma/np.sqrt(500):.2f}")
print()
print("Going from n=5 to n=30 cuts SE roughly in half.")
print("Going from n=100 to n=500 cuts SE by less than half that gain — the curve flattens.")

**Why n=30 is cited as a rule of thumb:**

- The CLT approximation becomes reasonable around n=30 for most moderately skewed distributions.
- The SE curve flattens noticeably after n=30, meaning the marginal benefit of each additional observation decreases rapidly.
- It is a pragmatic starting point, not a hard rule — highly skewed or heavy-tailed distributions may need n>50 or even n>100.

---
## 6. Hypothesis Testing: The Logic

### The Courtroom Analogy

> Just as a defendant is **innocent until proven guilty**, we assume the **null hypothesis is true until the data provide strong evidence against it**.

### Setting up hypotheses

| Hypothesis | Symbol | Meaning |
|---|---|---|
| Null hypothesis | H₀ | The "boring" status quo. No effect, no difference. |
| Alternative hypothesis | H₁ or Hₐ | What we suspect is actually true. |

**Example:**  
- H₀: The average restaurant bill is $18.00  
- H₁: The average restaurant bill is *not* $18.00

### The p-value

> **p-value** = the probability of observing data *at least as extreme as ours*, **assuming H₀ is true**.

- Small p-value (typically < 0.05) → the data are unlikely under H₀ → reject H₀
- Large p-value → the data are plausible under H₀ → fail to reject H₀

**Common misconception:** The p-value is NOT the probability that H₀ is true.

### Error types

| Decision \ Reality | H₀ is True | H₀ is False |
|---|---|---|
| Reject H₀ | **Type I Error (α)** — false positive | Correct ✓ |
| Fail to reject H₀ | Correct ✓ | **Type II Error (β)** — false negative |

The **significance level α** (usually 0.05) is the maximum Type I error rate we are willing to accept.

---
## 7. One-Sample t-Test

**Research question:** Does the average total bill at this restaurant differ from the industry average of $18.00?

- **H₀:** μ = 18.00 (mean bill equals industry average)
- **H₁:** μ ≠ 18.00 (mean bill differs from industry average)
- **α = 0.05**

In [ ]:
null_mean = 18.00
bills = tips['total_bill']

t_stat, p_value = stats.ttest_1samp(bills, popmean=null_mean)

print("One-Sample t-Test: total_bill vs claimed industry average")
print(f"  H₀: μ = ${null_mean:.2f}")
print(f"  H₁: μ ≠ ${null_mean:.2f}")
print()
print(f"  Sample mean:  x̄ = ${bills.mean():.2f}")
print(f"  t-statistic:  t = {t_stat:.4f}")
print(f"  p-value:      p = {p_value:.4f}")
print()
alpha = 0.05
if p_value < alpha:
    print(f"  p ({p_value:.4f}) < α ({alpha}) → REJECT H₀")
    print(f"  There IS sufficient evidence that the mean bill (${bills.mean():.2f}) ")
    print(f"  differs significantly from the claimed industry average of ${null_mean:.2f}.")
else:
    print(f"  p ({p_value:.4f}) ≥ α ({alpha}) → FAIL TO REJECT H₀")
    print(f"  There is NOT sufficient evidence to conclude the mean differs from ${null_mean:.2f}.")

In [ ]:
# Visualise: t-distribution with our observed t-statistic
df_val = len(bills) - 1
x = np.linspace(-5, 5, 400)
y = stats.t.pdf(x, df=df_val)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x, y, color='steelblue', linewidth=2.5, label='t-distribution (null)')

# Shade rejection regions (two-tailed α=0.05)
t_crit = stats.t.ppf(0.975, df=df_val)
x_left  = x[x <= -t_crit]
x_right = x[x >=  t_crit]
ax.fill_between(x_left,  stats.t.pdf(x_left,  df=df_val), alpha=0.3, color='crimson', label='Rejection region (α=0.05)')
ax.fill_between(x_right, stats.t.pdf(x_right, df=df_val), alpha=0.3, color='crimson')

# Mark the observed t-statistic
ax.axvline(t_stat, color='darkorange', linewidth=2.5, linestyle='--',
           label=f'Observed t = {t_stat:.2f}')

ax.set_xlabel('t-statistic')
ax.set_ylabel('Density')
ax.set_title(f'One-Sample t-Test  (p = {p_value:.4f})')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

---
## 8. Two-Sample t-Test

**Research question:** Do lunch and dinner customers spend different amounts on average?

- **H₀:** μ_lunch = μ_dinner (no difference in mean bill)
- **H₁:** μ_lunch ≠ μ_dinner
- **α = 0.05**

We also compute **Cohen's d** — a measure of *practical* significance (effect size), not just statistical significance.

In [ ]:
lunch  = tips.loc[tips['time'] == 'Lunch',  'total_bill']
dinner = tips.loc[tips['time'] == 'Dinner', 'total_bill']

t2, p2 = stats.ttest_ind(lunch, dinner)

# Cohen's d = (mean1 - mean2) / pooled_std
pooled_std = np.sqrt(((len(lunch)-1)*lunch.std()**2 + (len(dinner)-1)*dinner.std()**2) /
                     (len(lunch) + len(dinner) - 2))
cohens_d = (lunch.mean() - dinner.mean()) / pooled_std

print("Two-Sample t-Test: Lunch vs Dinner")
print(f"  Lunch   n={len(lunch):>3},  mean = ${lunch.mean():.2f}")
print(f"  Dinner  n={len(dinner):>3},  mean = ${dinner.mean():.2f}")
print()
print(f"  t-statistic: {t2:.4f}")
print(f"  p-value:     {p2:.4f}")
print(f"  Cohen's d:   {cohens_d:.4f}")
print()

if p2 < 0.05:
    print("  REJECT H₀ — significant difference between Lunch and Dinner bills.")
else:
    print("  FAIL TO REJECT H₀ — no significant difference detected.")

print()
print("Effect size guide: |d| < 0.2 small, 0.2–0.5 medium, > 0.8 large")
print(f"  |d| = {abs(cohens_d):.2f} → ", end="")
if abs(cohens_d) < 0.2:
    print("small practical effect")
elif abs(cohens_d) < 0.5:
    print("medium practical effect")
else:
    print("large practical effect")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
tips.boxplot(column='total_bill', by='time', ax=ax,
             boxprops=dict(color='steelblue'),
             medianprops=dict(color='crimson', linewidth=2))
ax.set_title(f'Total Bill by Time (p = {p2:.4f})')
ax.set_xlabel('Time of Day')
ax.set_ylabel('Total Bill ($)')
plt.suptitle('')  # Remove default 'Boxplot grouped by time' title
plt.tight_layout()
plt.show()

---
## 9. Chi-Square Test of Independence

The t-test compares *means* of numerical variables. When both variables are **categorical**, we use the **chi-square test**.

**Research question:** Is smoking status *independent* of meal time (Lunch vs Dinner)?

- **H₀:** Smoking status and meal time are *independent* (no relationship)
- **H₁:** Smoking status and meal time are *not* independent
- **α = 0.05**

In [ ]:
# Contingency table
contingency = pd.crosstab(tips['smoker'], tips['time'],
                           rownames=['Smoker'], colnames=['Time'])
print("Observed counts:")
print(contingency)
print()

chi2, p_chi2, dof, expected = stats.chi2_contingency(contingency)

print(f"Expected counts (if independent):")
print(pd.DataFrame(expected.round(1), index=contingency.index, columns=contingency.columns))
print()
print(f"Chi-square statistic: χ² = {chi2:.4f}")
print(f"Degrees of freedom:   df = {dof}")
print(f"p-value:              p  = {p_chi2:.4f}")
print()
if p_chi2 < 0.05:
    print("REJECT H₀ — there IS a significant association between smoking status and meal time.")
else:
    print("FAIL TO REJECT H₀ — no significant association detected between smoking status and meal time.")

In [ ]:
# Visualise as a grouped bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

contingency.plot(kind='bar', ax=axes[0], color=['#3498db', '#e74c3c'],
                 edgecolor='white', alpha=0.85)
axes[0].set_title('Observed counts: Smoker vs Time')
axes[0].set_xlabel('Smoker')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Row percentages to see relative proportions
contingency_pct = contingency.div(contingency.sum(axis=1), axis=0) * 100
contingency_pct.plot(kind='bar', ax=axes[1], color=['#3498db', '#e74c3c'],
                     edgecolor='white', alpha=0.85)
axes[1].set_title('Row %: Smoker vs Time\n(if independent, bars should be the same height)')
axes[1].set_xlabel('Smoker')
axes[1].set_ylabel('Percentage (%)')
axes[1].tick_params(axis='x', rotation=0)

plt.suptitle(f'Chi-Square Test  χ²={chi2:.2f},  p={p_chi2:.4f}', fontsize=12)
plt.tight_layout()
plt.show()

---
## 10. Practice Exercises

Apply what you have learned. Three exercises with increasing difficulty.

---

### Exercise 1 — Confidence Interval for Mean Tip

Construct a **95% confidence interval** for the mean *tip* amount in the dataset.

1. Treat `tips['tip']` as your sample.
2. Use `scipy.stats.t.interval()` to compute the CI.
3. Print the interval in plain English: "We are 95% confident the true mean tip is between $X and $Y."

In [ ]:
# Your code here


<details>
<summary>Solution (click to expand)</summary>

```python
tip_data = tips['tip']
n_tip  = len(tip_data)
lo, hi = stats.t.interval(0.95, df=n_tip-1, loc=tip_data.mean(), scale=stats.sem(tip_data))
print(f"95% CI for mean tip: (${lo:.2f}, ${hi:.2f})")
print(f"We are 95% confident the true mean tip is between ${lo:.2f} and ${hi:.2f}.")
```
</details>

---

### Exercise 2 — t-Test: Tips by Gender

Do male and female customers tip *different* amounts on average?

1. Split `tips['tip']` into Male and Female groups using `tips['sex']`.
2. Run a two-sample t-test with `scipy.stats.ttest_ind()`.
3. Print t-statistic, p-value, and your conclusion (reject / fail to reject H₀).

In [ ]:
# Your code here


<details>
<summary>Solution (click to expand)</summary>

```python
male_tips   = tips.loc[tips['sex'] == 'Male',   'tip']
female_tips = tips.loc[tips['sex'] == 'Female', 'tip']

t_sex, p_sex = stats.ttest_ind(male_tips, female_tips)
print(f"Male   mean tip: ${male_tips.mean():.2f}")
print(f"Female mean tip: ${female_tips.mean():.2f}")
print(f"t = {t_sex:.4f},  p = {p_sex:.4f}")
if p_sex < 0.05:
    print("Reject H₀ — significant difference in tips between genders.")
else:
    print("Fail to reject H₀ — no significant difference detected.")
```
</details>

---

### Exercise 3 — Chi-Square: Day vs Smoker

Is smoking status associated with the day of the week?

1. Build a contingency table with `pd.crosstab(tips['smoker'], tips['day'])`.
2. Run `scipy.stats.chi2_contingency()`.
3. Print the chi-square statistic, p-value, and conclusion.

In [ ]:
# Your code here


<details>
<summary>Solution (click to expand)</summary>

```python
ct_day = pd.crosstab(tips['smoker'], tips['day'])
print(ct_day)

chi2_day, p_day, dof_day, _ = stats.chi2_contingency(ct_day)
print(f"\nχ² = {chi2_day:.4f},  df = {dof_day},  p = {p_day:.4f}")
if p_day < 0.05:
    print("Reject H₀ — smoking status is associated with day of the week.")
else:
    print("Fail to reject H₀ — no significant association between smoking and day.")
```
</details>

---
## 11. Summary

| Concept | Key idea |
|---|---|
| Population vs Sample | We never see the full population; we reason from a sample |
| CLT | Sample means are approximately normal for n ≥ 30, regardless of population shape |
| Standard Error | SE = σ/√n; larger samples → smaller SE → more precise estimates |
| Confidence Interval | A range of plausible values for μ; wider = more confident |
| p-value | Probability of observing data this extreme *if H₀ were true* |
| One-sample t-test | Compare a sample mean to a known/claimed value |
| Two-sample t-test | Compare means of two independent groups |
| Chi-square test | Test independence between two categorical variables |

### Common traps to avoid

- **p < 0.05 does not mean the effect is large** — always check effect size (Cohen's d)
- **p-value ≠ probability that H₀ is true** — it is a conditional probability given H₀
- **Failing to reject H₀ ≠ proving H₀** — absence of evidence is not evidence of absence
- **Statistical significance ≠ practical significance** — a massive sample can detect tiny, meaningless differences

### Next steps

In **4.2 — Regression Analysis**, we move from comparing groups to modelling relationships between variables: how does bill size predict tip amount? How do multiple features together explain variation in an outcome?

You will build on everything here — hypothesis tests, p-values, and confidence intervals appear again as tools for evaluating regression coefficients.